# Topic emergenti — di cosa parlano *davvero*, senza liste nostre

I notebook 01 e 02 partono da **temi scelti da noi** (diamo le parole chiave e
misuriamo). Qui ribaltiamo: lasciamo che i temi **emergano dai testi**. Gli
embedding e5 dei ~25.000 documenti sono già nell'indice — li raggruppiamo
(**KMeans**) e diamo un nome a ogni gruppo con le sue parole più caratteristiche
(**c-TF-IDF**). È il primo passo della *topic extraction*.

L'ipotesi da verificare (la mappa che ci aspettiamo): una lunga **linea rossa**
liturgica e devozionale (Dio, Gesù, Spirito, Maria, i santi) che torna per tutti;
un grande strato **"dovuto"** dal ruolo (udienze a categorie, corpi, viaggi); e
pochi cluster dove un Papa lascia la **firma** (il suo programma, l'attualità).

## Come gira

Legge solo i **vettori già indicizzati** via `ponte.tabella()` — niente modello,
niente torch. Serve `scikit-learn` (vedi `../setup/`).

> ⚠️ **Ambiente.** `KMeans` di scikit-learn usa OpenMP: su alcuni env conda
> Windows con MKL/torch può crashare a livello nativo (doppio `libiomp5md.dll`).
> Se succede, gira in un ambiente "sano" (numpy non-MKL, es. `conda install
> nomkl`, o un env pip-only). Questo notebook è già **torch-free** apposta.

In [ ]:
import collections
import numpy as np
from ponte import tabella

tab = tabella()
t = tab.search().where("escludibile=false").select(
    ["url", "papa", "tipologia", "testo", "vector"]).limit(10**9).to_arrow()
urls = t.column("url").to_pylist(); papi = t.column("papa").to_pylist()
tipi = t.column("tipologia").to_pylist(); testi = t.column("testo").to_pylist()
V = np.stack(t.column("vector").to_pylist()).astype("float32")

# mean-pool dei chunk -> un vettore per documento (rinormalizzato)
idx = collections.defaultdict(list)
for i, u in enumerate(urls):
    idx[u].append(i)
docs = list(idx)
D = np.zeros((len(docs), V.shape[1]), "float32")
dpapa, dtipo, dtxt = [], [], []
for r, u in enumerate(docs):
    ii = idx[u]; v = V[ii].mean(0); D[r] = v / (np.linalg.norm(v) + 1e-9)
    dpapa.append(papi[ii[0]]); dtipo.append(tipi[ii[0]])
    dtxt.append(" ".join(" ".join(testi[i].split()) for i in ii)[:1200])
dpapa = np.array(dpapa)
print(len(docs), "documenti")

## KMeans sugli embedding + etichette c-TF-IDF

`K` gruppi sui vettori e5; per ogni gruppo le parole con la c-TF-IDF più alta
(frequenti *nel* gruppo ma rare *fra* i gruppi) fanno da etichetta. Per ogni topic
guardiamo la **dimensione**, il **lift per Papa** e la **tipologia** dominante (che
aiuta a riconoscere il "dovuto").

In [ ]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

PAPI = ["Papa Giovanni Paolo II", "Papa Benedetto XVI", "Papa Francesco", "Papa Leone XIV"]
AB = ["GP2", "BXVI", "FRA", "LEO"]
tot = collections.Counter(dpapa); N = len(docs)

K = 28
lab = KMeans(n_clusters=K, n_init=5, random_state=0).fit(D).labels_

# parole troppo generiche/di servizio: non fanno da etichetta
STOP = ("dell della delle degli dei del nel nella nelle negli nei alla alle agli "
        "dalla dalle dello sulla sulle perche perch egli essa esso quale quali quello "
        "quella quelle questi queste questo questa cari care caro carissimi fratelli "
        "sorelle signore signori signor amici saluto saluti saluta rivolgo ringrazio "
        "ringraziare grazie cordiale particolare gioia mondo vita anni anno giorno "
        "giorni oggi sempre molto bene quindi inoltre infatti nonche affinche presso "
        "desidero vorrei voluto presente numerosi occasione cosi nell siete sono tutti "
        "vostra vostro hanno copyright libreria editrice vaticana bollettino stampa "
        "trasmissioni televisivo video galleria diretta your vous para portoghese "
        "tedesco inglese spagnolo italiano francese lingua").split()
vec = TfidfVectorizer(stop_words=STOP, token_pattern=r"(?u)\b[a-zàèéìòù]{4,}\b",
                      max_features=4000, min_df=2, sublinear_tf=True)
clus = [" ".join(dtxt[i] for i in range(N) if lab[i] == c) for c in range(K)]
X = vec.fit_transform(clus); terms = np.array(vec.get_feature_names_out())

for c in sorted(range(K), key=lambda c: -(lab == c).sum()):
    n = int((lab == c).sum())
    top = terms[X[c].toarray().ravel().argsort()[::-1][:8]]
    lift = [(((lab == c) & (dpapa == p)).sum() / n) / (tot[p] / N) for p in PAPI]
    tp = collections.Counter(dtipo[i] for i in range(N) if lab[i] == c).most_common(1)[0]
    print(f"n={n:5}  " + " ".join(top))
    print("       lift " + " ".join(f"{AB[k]}{lift[k]:.1f}" for k in range(4)) + f"  | {tp[0]}")

## I topic emergenti (K=28) — raggruppati

I 28 gruppi si leggono benissimo nelle famiglie che ci aspettavamo.

**🔴 Linea rossa — liturgia e devozione** (il blocco più grande, lift ~1 per tutti
= continuità):

| topic | parole | nota |
|---|---|---|
| Cristo / fede / giovani / gioia | predicazione di base | n≈2000, tutti |
| Triduo / calice / Pilato / mistero | Passione, Pasqua | GP2 1.3 · BXVI 1.2 |
| Vangelo del giorno (scribi, farisei, parabole) | commento domenicale | **FRA 3.2 · LEO 2.4** |
| vespri / salmi / lodi / catechesi | liturgia delle ore | BXVI/FRA/LEO |
| Maria / Vergine / Elisabetta | mariano | tutti |
| presepe / Betlemme / Magi | Natale | BXVI 1.7 |

**🏛️ Il "dovuto" istituzionale** (udienze e ricorrenze dettate dal ruolo):

| topic | parole | nota |
|---|---|---|
| ad limina / episcopato / vescovi | visite dei vescovi | GP2 1.3 |
| capitolari / superiora / vita monastica | religiosi/e | GP2 1.3 |
| credenziali / ambasciatore / diplomatiche | corpo diplomatico | **BXVI 1.5** |
| comunale / Lazio / sindaco | autorità civili | GP2 1.3 |
| accademie / scienze / università | mondo accademico | BXVI/LEO |
| sanitari / malato / ospedale | sanità | tutti |
| Rota / coniugale / tribunale | diritto/matrimonio | GP2 1.2 |

**📜 Programma & storia / 📰 attualità** (qui "portano il loro"):

| topic | parole | nota |
|---|---|---|
| alimentazione / agricoltura / globalizzazione / *laudato* | sociale, FAO/UNESCO, ambiente | **FRA 1.4 · LEO 1.4** |
| Unitatis Redintegratio / Nostra Aetate / ecumenica | ecumenismo e dialogo | BXVI/LEO |
| Jasna Góra / Adalberto / Varsavia / Cracovia | **Polonia** | **GP2 1.5** |
| sport / calcio / squadra | sport | FRA 1.6 |
| nonni / giovani / *buonasera* | stile diretto di Francesco | **FRA 3.2** |

**✍️ Firme personali** · **✈️ viaggi**:

| topic | parole | nota |
|---|---|---|
| Santa Marta (omelie quotidiane) | *cotidie* | **FRA 4.1, solo lui** |
| aeroporto / volo / visita apostolica | viaggi | BXVI 2.3 |

## Cosa conferma — e il passo dopo

La mappa che ci aspettavamo **regge uno a uno**: la linea rossa liturgico-devozionale
è il grande continuo (lift ~1 per tutti); sopra ci sta lo strato **"dovuto"** dal
ruolo (vescovi, diplomatici, accademie, viaggi); e solo in pochi cluster un Papa
lascia la firma — **GP2** → Polonia e dottrina sociale del lavoro, **Francesco** →
sociale/*Laudato*, Santa Marta, stile diretto; **Leone** → dialogo e attualità.
Nessuna rottura: gli accenti cambiano, il filo resta.

Da qui i prossimi passi:
1. **pulizia** dei cluster di servizio (footer, bollettini, codici) — già attenuati
   dalle stopword, ma migliorabili;
2. il **"al netto del dovuto"**: togliere lo strato istituzionale/liturgico per far
   risaltare il profilo *scelto* di ciascun Papa;
3. legare i topic ai **mandati dichiarati** (omelie di inizio pontificato / primo
   saluto) e seguirli **nel tempo** — la domanda aperta nel README del repo.

---
*Solo viste **aggregate**: dimensioni dei gruppi, parole caratteristiche, lift per
Papa. Nessun testo ripubblicato (© Libreria Editrice Vaticana, fonte `url` nei
metadati).*